# Aula 5: Random & Pipeline

## Full ABT

Criamos a ABT com todos os clientes e todas as variáveis, contudo temos que pegar uma amostra de cada `DtRef`

In [0]:
%sql

WITH tb_ativacao AS (
    SELECT distinct IdCliente,
           date(DtCriacao) AS dtAtivacao
    FROM workspace.tmw_loyalty.transacoes
),

tb_resposta AS (

    SELECT t1.IdCliente,
           t1.dtRef,
           count(t2.dtativacao) AS qtdAtivacoes,
           max(case when t2.dtativacao is not null then 1 else 0 end) as flAtivacao

    FROM workspace.analytics_rm.fs_loyalty AS t1

    LEFT JOIN tb_ativacao AS t2
    ON t1.IdCliente = t2.idcliente
    AND t1.dtRef <= t2.dtAtivacao
    AND t2.dtAtivacao <= (t1.dtRef + INTERVAL 28 DAYS)

    GROUP BY ALL
    ORDER BY t1.IdCliente, dtRef
),

tb_features AS ( 

    SELECT t2.*,
        t3.qtdCursosIniciados,
        t3.qtdCursosFinalizados,
        t3.python_2025,
        t3.plataforma_ml_2026,
        t3.mlflow_2025,
        t3.carreira,
        t3.nekt_2025,
        t3.estatistica_2025,
        t3.coleta_dados_2024,
        t3.python_2024,
        t3.lago_mago_2024,
        t3.github_2025,
        t3.trampar_lakehouse_2024,
        t3.ds_databricks_2024,
        t3.sql_2020,
        t3.ds_pontos_2024,
        t3.streamlit_2025,
        t3.estatistica_2024,
        t3.ml_2024,
        t3.ragia,
        t3.sql_2025,
        t3.loyalty_predict_2025,
        t3.go_2026,
        t3.f1_lake,
        t3.ia_canal_2025,
        t3.tse_analytics_2024,
        t3.machine_learning_2025,
        t3.matchmaking_trampar_de_casa_2024,
        t3.pandas_2024,
        t3.pandas_2025,
        t3.speed_f1,
        t3.github_2024,
        t3.avgTempoInicioUltimo,
        t3.avgTempoInicioFim,
        t1.flAtivacao

    FROM tb_resposta AS t1

    LEFT JOIN workspace.analytics_rm.fs_loyalty AS t2
    ON t1.idcliente = t2.IdCliente
    AND t1.dtref=t2.dtRef

    LEFT JOIN workspace.analytics_rm.fs_education AS t3
    ON t1.idcliente = t3.IdCliente
    AND t1.dtref=t3.dtRef

) ,

tb_test_clientes AS (
    SELECT 
    dtRef,
    COUNT(*) AS qtdClientes
    FROM tb_features
    GROUP BY ALL
) ,

tb_test_ativacao AS (
    SELECT 
        avg(flativacao)
    FROM tb_features

)

SELECT 
    *
-- FROM tb_features
-- FROM tb_test_clientes
FROM tb_test_ativacao
GROUP BY ALL;


## Amostras Randomicas

Criamos primeiro a `tb_sample_old`, onde vamos randomizar a safra do cliente e pegar o primeiro.

Criamos então `tb_sample`, onde vamos pensar na OOT e deixar essa safra completa para termos a mesma taxa de resposta original enquanto que as safras anterioser serão randomizadas por cliente.
- Até a penultima safra faço a randomização.
- Na última safra pego a base completa.




In [0]:
%sql

WITH tb_ativacao AS (
    SELECT distinct IdCliente,
           date(DtCriacao) AS dtAtivacao
    FROM workspace.tmw_loyalty.transacoes
),

tb_resposta AS (

    SELECT t1.IdCliente,
           t1.dtRef,
           count(t2.dtativacao) AS qtdAtivacoes,
           max(case when t2.dtativacao is not null then 1 else 0 end) as flAtivacao

    FROM workspace.analytics_rm.fs_loyalty AS t1

    LEFT JOIN tb_ativacao AS t2
    ON t1.IdCliente = t2.idcliente
    AND t1.dtRef <= t2.dtAtivacao
    AND t2.dtAtivacao <= (t1.dtRef + INTERVAL 28 DAYS)

    GROUP BY ALL
    ORDER BY t1.IdCliente, dtRef
),

tb_features AS ( 

    SELECT t2.*,
        t3.qtdCursosIniciados,
        t3.qtdCursosFinalizados,
        t3.python_2025,
        t3.plataforma_ml_2026,
        t3.mlflow_2025,
        t3.carreira,
        t3.nekt_2025,
        t3.estatistica_2025,
        t3.coleta_dados_2024,
        t3.python_2024,
        t3.lago_mago_2024,
        t3.github_2025,
        t3.trampar_lakehouse_2024,
        t3.ds_databricks_2024,
        t3.sql_2020,
        t3.ds_pontos_2024,
        t3.streamlit_2025,
        t3.estatistica_2024,
        t3.ml_2024,
        t3.ragia,
        t3.sql_2025,
        t3.loyalty_predict_2025,
        t3.go_2026,
        t3.f1_lake,
        t3.ia_canal_2025,
        t3.tse_analytics_2024,
        t3.machine_learning_2025,
        t3.matchmaking_trampar_de_casa_2024,
        t3.pandas_2024,
        t3.pandas_2025,
        t3.speed_f1,
        t3.github_2024,
        t3.avgTempoInicioUltimo,
        t3.avgTempoInicioFim,
        t1.flAtivacao

    FROM tb_resposta AS t1

    LEFT JOIN workspace.analytics_rm.fs_loyalty AS t2
    ON t1.idcliente = t2.IdCliente
    AND t1.dtref=t2.dtRef

    LEFT JOIN workspace.analytics_rm.fs_education AS t3
    ON t1.idcliente = t3.IdCliente
    AND t1.dtref=t3.dtRef

),

tb_sample_old AS (

    SELECT *
    FROM tb_features
    QUALIFY row_number() OVER (PARTITION BY idCliente ORDER BY rand()) = 1

),

tb_sample AS (

    SELECT *
    FROM tb_features
    WHERE dtRef <= '2026-05-01'
    QUALIFY row_number() OVER (PARTITION BY idCliente ORDER BY rand()) = 1

    UNION ALL

    SELECT *
    FROM tb_features
    WHERE dtRef = '2026-06-01'
),

tb_test_clientes AS (
    SELECT 
    dtRef,
    COUNT(*) AS qtdClientes
    -- FROM tb_sample_old
    FROM tb_sample
    GROUP BY ALL
),

tb_test_ativacao AS (
    SELECT 
        avg(flativacao)
    -- FROM tb_sample_old
    FROM tb_sample

),

tb_test_ativacaoRef AS (
    SELECT 
        dtRef,
        COUNT(*) AS qtdClientes,
        avg(flativacao)
    FROM tb_sample
    GROUP BY ALL

)

SELECT 
    *
-- FROM tb_sample
-- FROM tb_test_clientes
-- FROM tb_test_ativacao
FROM tb_test_ativacaoRef
;

## Comparação FULL vs RANDOM ABT

No geral, a proporção não mudou muito (o que era grande continua grande, e tivemos algumas diferenças pontuais).

| FULL ABT |  |  |  | RANDOM ABT |  | DELTA |
| --- | --- | --- | --- | --- | --- | --- |
| dtRef | qtdClientes | pct | dtRef | qtdClientes | pct | DELTA |
| 2025-01-01 | 202 | 3.35% | 2025-01-01 | 63 | 2.81% | -0.54% |
| 2025-02-01 | 374 | 6.20% | 2025-02-01 | 168 | 7.50% | 1.30% |
| 2025-03-01 | 451 | 7.48% | 2025-03-01 | 225 | 10.04% | 2.56% |
| 2025-04-01 | 303 | 5.02% | 2025-04-01 | 101 | 4.51% | -0.52% |
| 2025-05-01 | 264 | 4.38% | 2025-05-01 | 68 | 3.03% | -1.34% |
| 2025-06-01 | 299 | 4.96% | 2025-06-01 | 107 | 4.77% | -0.18% |
| 2025-07-01 | 281 | 4.66% | 2025-07-01 | 74 | 3.30% | -1.36% |
| 2025-08-01 | 266 | 4.41% | 2025-08-01 | 98 | 4.37% | -0.04% |
| 2025-09-01 | 725 | 12.02% | 2025-09-01 | 403 | 17.98% | 5.97% |
| 2025-10-01 | 571 | 9.46% | 2025-10-01 | 209 | 9.33% | -0.14% |
| 2025-11-01 | 418 | 6.93% | 2025-11-01 | 141 | 6.29% | -0.64% |
| 2025-12-01 | 327 | 5.42% | 2025-12-01 | 93 | 4.15% | -1.27% |
| 2026-01-01 | 258 | 4.28% | 2026-01-01 | 73 | 3.26% | -1.02% |
| 2026-02-01 | 283 | 4.69% | 2026-02-01 | 77 | 3.44% | -1.25% |
| 2026-03-01 | 285 | 4.72% | 2026-03-01 | 94 | 4.19% | -0.53% |
| 2026-04-01 | 264 | 4.38% | 2026-04-01 | 89 | 3.97% | -0.40% |
| 2026-05-01 | 223 | 3.70% | 2026-05-01 | 71 | 3.17% | -0.53% |
| 2026-06-01 | 239 | 3.96% | 2026-06-01 | 87 | 3.88% | -0.08% |


### O que estamos tentando resolver pegando somente algumas safras dos clientes?
A suposição para a criação de cada modelo é que a base é IID (Independente e Identicamente Distribuida), e com multiplas safras podemos ter o mesmo individuo no treino e no teste. Então se o modelo aprende com um cliente e esse mesmo cliente vai estar na base de teste, o modelo tende a acertar mais só porque o mesmo individuo está na base de teste, por mais que as features sejam diferentes.

O que se pode ser feito:

1. O que foi proposto agora, cada indivíduo só entra uma vez na ABT.
    - **Atenção**: Pode haver um ruído da variável resposta sim, mas nada muito "grave" é esperado

2. Ou na hora em que vamos fazer o split de treino e teste, podemos fazer uma amostra por indivíduo. Ou seja, indivíduos que só aparecem na base de treino só vao ficar na base de treino, e outros individuos vão para a base de teste.
    - **Atenção**: O que pode acontecer aqui é, clientes que aparecem muitas vezes podem ficar na base de treino e os que aparecem pouuco fican na base de teste. O que não nos garantiria um split de 80/20 entre treino e teste. Nos dando um controle menor dos dados que estão no treino e teste.



### Taxa de Resposta

Comparando a taxa de resposta das duas ABTs:

| ABT | `avg(flAtivacao)` |
| --- | --- |
| FULL | 0.5193104591413891 |
| RANDOM | 0.2749546279491833 |

Caiu praticamente a metade, o que é um problema, já que não estamos mantendo a taxa de churn da base original.

O que pode ser feito para garantimos a mesma proporção, ou similar?
- Pegar a OOT com  abase completa, sem pegar uma unica safra de um cliente.
- Comparar se a taxa de resposta da OOT é parecida com a da ABT FULL.

| FULL ABT |  |  |  | RANDOM ABT |  |  | DELTA |
| --- | --- | --- | --- | --- | --- | --- | --- |
| dtRef | qtdClientes | pct | dtRef | qtdClientes | pct | taxa | DELTA |
| 2025-01-01 | 202 | 3.35% | 2025-01-01 | 67 | 2.99% | 38.81% | -0.36% |
| 2025-02-01 | 374 | 6.20% | 2025-02-01 | 166 | 7.41% | 24.70% | 1.21% |
| 2025-03-01 | 451 | 7.48% | 2025-03-01 | 205 | 9.15% | 12.68% | 1.67% |
| 2025-04-01 | 303 | 5.02% | 2025-04-01 | 108 | 4.82% | 30.56% | -0.20% |
| 2025-05-01 | 264 | 4.38% | 2025-05-01 | 69 | 3.08% | 33.33% | -1.30% |
| 2025-06-01 | 299 | 4.96% | 2025-06-01 | 86 | 3.84% | 18.60% | -1.12% |
| 2025-07-01 | 281 | 4.66% | 2025-07-01 | 93 | 4.15% | 27.96% | -0.51% |
| 2025-08-01 | 266 | 4.41% | 2025-08-01 | 95 | 4.24% | 20.00% | -0.17% |
| 2025-09-01 | 725 | 12.02% | 2025-09-01 | 397 | 17.72% | 30.48% | 5.70% |
| 2025-10-01 | 571 | 9.46% | 2025-10-01 | 215 | 9.59% | 33.49% | 0.13% |
| 2025-11-01 | 418 | 6.93% | 2025-11-01 | 140 | 6.25% | 26.43% | -0.68% |
| 2025-12-01 | 327 | 5.42% | 2025-12-01 | 88 | 3.93% | 31.82% | -1.49% |
| 2026-01-01 | 258 | 4.28% | 2026-01-01 | 68 | 3.03% | 25.00% | -1.24% |
| 2026-02-01 | 283 | 4.69% | 2026-02-01 | 93 | 4.15% | 27.96% | -0.54% |
| 2026-03-01 | 285 | 4.72% | 2026-03-01 | 92 | 4.11% | 35.87% | -0.62% |
| 2026-04-01 | 264 | 4.38% | 2026-04-01 | 97 | 4.33% | 29.90% | -0.05% |
| 2026-05-01 | 223 | 3.70% | 2026-05-01 | 75 | 3.35% | 38.67% | -0.35% |
| 2026-06-01 | 239 | 3.96% | 2026-06-01 | 239 | 10.66% | 56.07% | 6.70% |


| ABT | `avg(flAtivacao)` |
| --- | --- |
| FULL | 0.5193104591413891 |
| RANDOM - OOT | 0.560669456066945 |

E assim  consegimos garantir que quando rodarmos o modelo da OOT, o resultado esperado das metricas do modelo são comparáveis a realidade da base original.
Teriamos um problema se o modelo fosse treinando com uma média diferente do target, por isso temos a OOT. 

> A base OOT (back test) vai dizer a real performance do modelo em produção.

O mais fiel possivel seria pegar a OOT com clientes que o modelo nunca viu, ou seja, totalmente novos. Mas como já fizemos a randomização anterior, esse erro começa a ficar mais diluido.


## Criação da ABT

Gero a base `workspace.analytics_rm.abt_ativacao` através do arquivo `. > Modelo > 2_ABT_RM`.



## Pipeline

Vamos começar a pensar no pipeline:
- Temos que fazer o split Treino/Teste
- Avaliar o que fazer com os missings

### SEMMA - S: Sample
 

In [0]:
# SAMPLE

from sklearn import model_selection

df_abt = spark.table("workspace.analytics_rm.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef'] == df_abt['dtRef'].max()] # <- usa somente a últma safra como back test (OOT)

df_train_test = df_abt[df_abt['dtRef'] != df_abt['dtRef'].max()] # <- usa toda a base para ser separada entre treino e teste sem a OOT

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")


### SEMMA - E: Explore

Vamos analisar os missings da base e definir o que seria melhor fazer com eles.



In [0]:
# EXPLORE

is_na = X_train.isna().mean()
is_na[is_na>0]

# DiasUltimoStreak -> 28 dias

# qtdCursosIniciados -> 0
# qtdCursosFinalizados -> 0
# python_2025 -> 0
# plataforma_ml_2026 -> 0
# mlflow_2025 -> 0
# carreira -> 0
# nekt_2025 -> 0
# estatistica_2025 -> 0
# coleta_dados_2024 -> 0
# python_2024 -> 0
# lago_mago_2024 -> 0
# github_2025 -> 0
# trampar_lakehouse_2024 -> 0
# ds_databricks_2024 -> 0
# sql_2020 -> 0
# ds_pontos_2024 -> 0
# streamlit_2025 -> 0
# estatistica_2024 -> 0
# ml_2024 -> 0
# ragia -> 0
# sql_2025 -> 0
# loyalty_predict_2025 -> 0
# go_2026 -> 0
# f1_lake -> 0
# ia_canal_2025 -> 0
# tse_analytics_2024 -> 0
# machine_learning_2025 -> 0
# matchmaking_trampar_de_casa_2024 -> 0
# pandas_2024 -> 0
# pandas_2025 -> 0
# speed_f1 -> 0
# github_2024 -> 0

# avgTempoInicioUltimo -> max base
# avgTempoInicioFim -> max base

In [0]:
# EXPLORE
# Verificando qual o máximo das variáveis da base de treino.
# Note que os dias depois de o último streak = 28, então ao inves de imputar 28, posso usar o máximo da base também.
X_train[is_na[is_na>0].index].describe().T


Note que temos bastante missings, principalmente nas **features de education**.

`diasDesdeUltimoStreak`: é calculada dentro da janela, ou seja o máximo aqui seria 28 dias. Como essa variável, o quão menor mais recente o streak, quando for missing, teriamos que ter o maior valor da janela.

`avgTempoInicioUltimo` e `avgTempoInicioFim`: é calculada fora da janela, ou seja o máximo aqui seria o máximo da base. Isso porque se a pessoa não está engajada, ela vai levar muito tempo apra completar os cursos.

Todas as de cusro vão receber 0.




| Feature                          | pct Missing | Comment                                                                                                                                                                         | Replace with |
| -------------------------------- | ----------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------ |
| pctPontosAbs_streamElements      | 0.980847    |                                                                                                                                                                                 | 0            |
| pctPontosAbs_ponei               | 0.977365    |                                                                                                                                                                                 | 0            |
| pctPontosAbs_streak              | 0.921648    |                                                                                                                                                                                 | 0            |
| diasDesdeUltimoStreak            | 0.921648    | Quanto menor, mais recente foi o streak, então faz sentido colocar un número alto.<br>Como essa variável foi calculada dentro da janela, 28 dias seria o maior número possível. | max da base |
| pctPontosAbs_outros              | 0.917586    |                                                                                                                                                                                 | 0            |
| avgTempoInicioFim                | 0.882182    |                                                                                                                                                                                 | max base     |
| qtdCursosIniciados               | 0.844457    |                                                                                                                                                                                 | 0            |
| qtdCursosFinalizados             | 0.844457    |                                                                                                                                                                                 | 0            |
| python_2025                      | 0.844457    |                                                                                                                                                                                 | 0            |
| plataforma_ml_2026               | 0.844457    |                                                                                                                                                                                 | 0            |
| mlflow_2025                      | 0.844457    |                                                                                                                                                                                 | 0            |
| carreira                         | 0.844457    |                                                                                                                                                                                 | 0            |
| nekt_2025                        | 0.844457    |                                                                                                                                                                                 | 0            |
| estatistica_2025                 | 0.844457    |                                                                                                                                                                                 | 0            |
| coleta_dados_2024                | 0.844457    |                                                                                                                                                                                 | 0            |
| python_2024                      | 0.844457    |                                                                                                                                                                                 | 0            |
| lago_mago_2024                   | 0.844457    |                                                                                                                                                                                 | 0            |
| github_2025                      | 0.844457    |                                                                                                                                                                                 | 0            |
| trampar_lakehouse_2024           | 0.844457    |                                                                                                                                                                                 | 0            |
| ds_databricks_2024               | 0.844457    |                                                                                                                                                                                 | 0            |
| sql_2020                         | 0.844457    |                                                                                                                                                                                 | 0            |
| ds_pontos_2024                   | 0.844457    |                                                                                                                                                                                 | 0            |
| streamlit_2025                   | 0.844457    |                                                                                                                                                                                 | 0            |
| estatistica_2024                 | 0.844457    |                                                                                                                                                                                 | 0            |
| ml_2024                          | 0.844457    |                                                                                                                                                                                 | 0            |
| ragia                            | 0.844457    |                                                                                                                                                                                 | 0            |
| sql_2025                         | 0.844457    |                                                                                                                                                                                 | 0            |
| loyalty_predict_2025             | 0.844457    |                                                                                                                                                                                 | 0            |
| go_2026                          | 0.844457    |                                                                                                                                                                                 | 0            |
| f1_lake                          | 0.844457    |                                                                                                                                                                                 | 0            |
| ia_canal_2025                    | 0.844457    |                                                                                                                                                                                 | 0            |
| tse_analytics_2024               | 0.844457    |                                                                                                                                                                                 | 0            |
| machine_learning_2025            | 0.844457    |                                                                                                                                                                                 | 0            |
| matchmaking_trampar_de_casa_2024 | 0.844457    |                                                                                                                                                                                 | 0            |
| pandas_2024                      | 0.844457    |                                                                                                                                                                                 | 0            |
| pandas_2025                      | 0.844457    |                                                                                                                                                                                 | 0            |
| speed_f1                         | 0.844457    |                                                                                                                                                                                 | 0            |
| github_2024                      | 0.844457    |                                                                                                                                                                                 | 0            |
| avgTempoInicioUltimo             | 0.844457    |                                                                                                                                                                                 | max base     |
| pctPontosAbs_chat                | 0.386535    |                                                                                                                                                                                 | 0            |
| pctPontosAbs_presenca            | 0.200232    |                                                                                                                                                                                 | 0            |







In [0]:
# MODIFY

!pip install feature_engine


In [0]:
# MODIFY

from feature_engine import imputation

features_0 = [
    'pctPontosAbs_streamElements',
    'pctPontosAbs_ponei',
    'pctPontosAbs_streak',
    'pctPontosAbs_outros',
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'python_2025',
    'plataforma_ml_2026',
    'mlflow_2025',
    'carreira',
    'nekt_2025',
    'estatistica_2025',
    'coleta_dados_2024',
    'python_2024',
    'lago_mago_2024',
    'github_2025',
    'trampar_lakehouse_2024',
    'ds_databricks_2024',
    'sql_2020',
    'ds_pontos_2024',
    'streamlit_2025',
    'estatistica_2024',
    'ml_2024',
    'ragia',
    'sql_2025',
    'loyalty_predict_2025',
    'go_2026',
    'f1_lake',
    'ia_canal_2025',
    'tse_analytics_2024',
    'machine_learning_2025',
    'matchmaking_trampar_de_casa_2024',
    'pandas_2024',
    'pandas_2025',
    'speed_f1',
    'github_2024',
    'pctPontosAbs_chat',
    'pctPontosAbs_presenca',

]

# DEFINIÇ
impute_0 = imputation.ArbitraryNumberImputer(variables=features_0, arbitrary_number=0)
# impute_28 = imputation.ArbitraryNumberImputer(variables='DiasUltimoStreak', arbitrary_number=28)  # <- Não preciso mais desse, e vou coloar tudo no impute max tail

impute_max_tail = imputation.EndTailImputer(variables=['diasDesdeUltimoStreak','avgTempoInicioUltimo','avgTempoInicioFim'],
                                           imputation_method='max', fold=1)



In [0]:
# AJUSTE DOS MÉTODOS
# Preciso ajustar os métodos para que ele saiba quais são os números da base de treino que vamos usar para 
impute_0.fit(X_train)
impute_max_tail.fit(X_train)


In [0]:
impute_max_tail.fit(X_train).imputer_dict_  # Mostra os valores que foram usados

In [0]:
# APLICAÇÃO DOS MÉTODOS
X_train_transform = impute_0.transform(X_train)
X_train_transform = impute_max_tail.transform(X_train_transform)

X_train_transform.isna().sum().sort_values(ascending=False)  # <- mostrando que não tem mais missing na base

In [0]:
# EXPLORE 2
# Quero fazer uma análise bivariada quais as features que tem alteração na minha classe
import pandas as pd
pd.set_option('display.max_rows', 100)
import numpy as np

df_train = X_train_transform.copy()
# df_train[y_train.name] = y_train  # fica com o nome original da target
df_train["target"] = y_train  # renomeio para target

# df_train

sumario = df_train.groupby("target").mean().T
sumario['rate'] = sumario[1].div(sumario[0].where(sumario[0] != 0, np.nan))
sumario['rate'] = sumario['rate'].fillna(0)
sumario
# print(sumario.sort_values(by='rate', ascending=False))


Explicação:


| Feature | target = 0 | target = 1 | rate | 
| --- | --- | --- | --- |
| QtdeFrequencia | 1.740711 | 4.863918 | 2.794213 |

- **QtdeFrequencia**
    - um rate the 2.79 é bem expressivo, significando que houve uma mudança de mais de 2 vezes de um grupo para o outro.


| Feature | target = 0 | target = 1 | rate | 
| --- | --- | --- | --- |
| zScore | -0.26164 | 0.056697 | -0.216700 |

- **zScore**: pontos positivos
    - quem é 1 está do lado direito da normal e quem é 0 está muito negativo
    - temos que tomar cuidado pois ela pode ter multicolinearidade com a quantidade de pontos!


| Feature | target = 0 | target = 1 | rate | 
| --- | --- | --- | --- |
| pctTransacao_chat | 0.421985 | 0.528327 | 1.252005 |
| pctTransacao_presenca | 0.479552 | 0.402149 | 0.838592 |
| pctTransacao_streak | 0.001988 | 0.012966 | 6.522597 |
| pctTransacao_ponei | 0.001132 | 0.008199 | 7.243653 |
| pctTransacao_streamElements | 0.000056 | 0.00182 | 32.593147 |
| pctTransacao_outros | 0.095287 | 0.046539 | 0.488408 |

- pctTransacao
    - Temos um aumento de 10pp de churn dado que a pessoa nao participa do chat (0.42 → 0.52)
    - Se a pessoa assina a lista de presença tem quase 8pp (0.47 → 0.40)

